In [ ]:
import math
import torch
import gpytorch
from matplotlib import pyplot as plt
import h5py
import numpy as np
import random
import bacco

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

##### Get the cosmological parameters

In [ ]:
wind_en      = []
wind_vel     = []
rho_rec      = []
sf_ts        = []
ef_kin       = []
ef_high      = []
f_re         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re.append(float(line.split()[1]))
        
wind_en   = np.asarray(wind_en)
wind_vel  = np.asarray(wind_vel)
rho_rec   = np.asarray(rho_rec)
sf_ts     = np.asarray(sf_ts)
ef_kin    = np.asarray(ef_kin)
ef_high   = np.asarray(ef_high)
f_re      = np.asarray(f_re)

In [ ]:
def pars(i, mstar):

    arr = np.vstack( (mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * rho_rec[i],\
                        np.ones(len(mstar)) * sf_ts[i],\
                        np.ones(len(mstar)) * ef_kin[i],\
                        np.ones(len(mstar)) * ef_high[i],\
                        np.ones(len(mstar)) * f_re[i])).T

    return arr

In [ ]:
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/MTNG/", snap=264)
mtng.fof['halo_pos'][:,0] = ( mtng.fof['halo_pos'][:,0] - 125 ) % 500

In [ ]:
with open("/lscratch/fgmaion/MTNG-resims/halo_selection/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

### Get the zoooms

In [ ]:
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = {}

loaded = []
for i in range(30):
    try:
        base = "/cosmos_storage/data_sharing/MN5_resims/LH_{:d}/hydro_output/".format(i)
        zoom[i] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                                tau=tau, ns=ns, sigma8=sigma8, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap), use_ids=True, numpart=4320)
        loaded.append(i)
    except:
        print("Failed for sim {:d}".format(i))

In [ ]:
xmatch = {}

for i in loaded:
    xmatch[i] = utils.cross_match(zoom[i], snap=264)

In [ ]:
m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

In [ ]:
zoom_split = {}
zoom_sel = {}
for i in loaded:
    zoom_split[i] = utils.split_halos(zoom[i])

    zoom_sel[i] = {}

    zoom_sel[i]['sel'] = xmatch[i]['ind'][:,np.newaxis,np.newaxis]
    zoom_sel[i]['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
zoom_smf = {}

Nbins = 20

for i in loaded:
    zoom_smf[i] = zoom_split[i].halo_smf(sel_mask=zoom_sel[i], nbins=Nbins, draws=1)

In [ ]:
train_sel = random.sample(loaded, 12)
test_sel  = np.delete(loaded, np.isin(loaded, train_sel))

In [ ]:
#del smf_global, arr_global
mstar = np.log10(zoom_smf[train_sel[0]]['mstar'][0][:-3])
arr_global = pars(0, mstar)
smf_global = np.log10(zoom_smf[train_sel[0]]['smf'][0][:-3])

for i in range(len(train_sel)):
    mstar = np.log10(zoom_smf[train_sel[i]]['mstar'][0][:-3])

    arr = pars(train_sel[i], mstar)

    arr_global = np.vstack((arr_global, arr))

    smf_global = np.hstack((smf_global, np.log10(zoom_smf[train_sel[i]]['smf'][0][:-3])))

isnan = np.where(np.isnan(arr_global))[0]

arr_global = np.delete(arr_global, isnan, axis=0)
smf_global = np.delete(smf_global, isnan)

In [ ]:
#Training data is 100 points in [0,1] inclusive regularly spaced
#train_x = torch.vstack([torch.linspace(0, 1, 100),torch.linspace(0, 1, 100)]).T
#True function is sin(2*pi*x) with Gaussian noise
#train_y = torch.sin(train_x[:,0] * (2 * math.pi)) + torch.randn(train_x[:,0].size()) * math.sqrt(0.04)

In [ ]:
train_x = torch.asarray(arr_global, dtype=torch.float)
train_y = torch.asarray(smf_global, dtype=torch.float)

### Build the GP Model

In [ ]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=8))

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(train_x, train_y, likelihood)

In [ ]:
# # We will use the simplest form of GP model, exact inference
# class SpectralMixtureGPModel(gpytorch.models.ExactGP):
#     def __init__(self, train_x, train_y, likelihood):
#         super(SpectralMixtureGPModel, self).__init__(train_x, train_y, likelihood)
#         self.mean_module = gpytorch.means.ConstantMean()
#         self.covar_module = gpytorch.kernels.SpectralMixtureKernel(num_mixtures=4)
#         #self.covar_module.initialize_from_data(train_x, train_y)

#     def forward(self, x):
#         mean_x = self.mean_module(x)
#         covar_x = self.covar_module(x)
#         return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# # initialize likelihood and model
# likelihood = gpytorch.likelihoods.GaussianLikelihood()
# model = SpectralMixtureGPModel(train_x, train_y, likelihood)

#### Train it

In [ ]:
# this is for running the notebook in our testing framework
import os
smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 200


# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(train_x)
    # Calc loss and backprop gradients
    loss = -torch.sum(mll(output, train_y))
    loss.backward()
    print('Iter %d/%d - Loss: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.likelihood.noise.item()
    ))
    optimizer.step()

In [ ]:
mstar_test = np.linspace(8,11,15)
test_x1 = torch.asarray(pars(train_sel[1], np.log10(zoom_smf[train_sel[1]]['mstar'][0][:-5])), dtype=torch.float)
test_x2 = torch.asarray(pars(test_sel[0], np.log10(zoom_smf[test_sel[0]]['mstar'][0][:-5])), dtype=torch.float)

In [ ]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

# Test points are regularly spaced along [0,1]
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    observed_pred1 = likelihood(model(test_x1))
    observed_pred2 = likelihood(model(test_x2))

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 1, figsize=(5.5, 5), dpi=150)

    # Get upper and lower confidence bounds
    lower1, upper1 = observed_pred1.confidence_region()
    lower2, upper2 = observed_pred2.confidence_region()
    # Plot training data as black stars
    ax.plot(np.log10(zoom_smf[train_sel[1]]['mstar'][0][:-5]), np.log10(zoom_smf[train_sel[1]]['smf'][0][:-5]), 'b*', label='Train Set')
    ax.plot(np.log10(zoom_smf[test_sel[0]]['mstar'][0][:-5]), np.log10(zoom_smf[test_sel[0]]['smf'][0][:-5]), 'r*', label='Test Set')

    # Plot predictive means as blue line
    ax.plot(test_x1[:,0], observed_pred1.mean.numpy(), 'b')
    ax.plot(test_x2[:,0], observed_pred2.mean.numpy(), 'r')

    # Shade between the lower and upper confidence bounds
    ax.fill_between(test_x1.numpy()[:,0], observed_pred1.mean.numpy()-observed_pred1.stddev.numpy(), observed_pred1.mean.numpy()+observed_pred1.stddev.numpy(), color='b', alpha=0.5, edgecolor=None)
    ax.fill_between(test_x2.numpy()[:,0], observed_pred2.mean.numpy()-observed_pred2.stddev.numpy(), observed_pred2.mean.numpy()+observed_pred2.stddev.numpy(), color='r', alpha=0.5, edgecolor=None)

ax.legend()

ax.set_xlabel('$M_*$')
ax.set_ylabel('SMF')


In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150, sharey=True)
    
    plt.subplots_adjust(wspace=0, hspace=0)

    for i in range(40):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x1 = torch.asarray(pars(train_sel[i], np.log10(smf[train_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred1 = likelihood(model(test_x1))
    
        ax[0].plot(np.log10(smf[train_sel[i]]['mstar'][:-5]),  np.log10(smf[train_sel[i]]['smf'][:-5]) - observed_pred1.mean.numpy(), color='b', lw=0.7)
        ax[0].fill_between(np.log10(smf[train_sel[i]]['mstar'][:-5]), np.log10(smf[train_sel[i]]['smf'][:-5]) - (observed_pred1.mean.numpy()-observed_pred1.stddev.numpy()) , np.log10(smf[train_sel[i]]['smf'][:-5]) - (observed_pred1.mean.numpy()+observed_pred1.stddev.numpy()), alpha=0.05, color='b', edgecolor=None)

    for i in range(60):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x2 = torch.asarray(pars(test_sel[i], np.log10(smf[test_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred2 = likelihood(model(test_x2))
        
        ax[1].plot(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - observed_pred2.mean.numpy(), color='r', lw=0.7)
        ax[1].fill_between(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - (observed_pred2.mean.numpy()-observed_pred2.stddev.numpy()) , np.log10(smf[test_sel[i]]['smf'][:-5]) - (observed_pred2.mean.numpy()+observed_pred1.stddev.numpy()), alpha=0.05, color='r', edgecolor=None)
#        ax[1].fill_between(np.log10(smf[test_sel[i]]['mstar'][:-5]), np.log10(smf[test_sel[i]]['smf'][:-5]) - lower2.numpy(), np.log10(smf[test_sel[i]]['smf'][:-5]) - upper2.numpy(), alpha=0.01, color='r', edgecolor=None)

    # Plot training data as black stars
    ax[0].axhline(0, color='k', ls='--')
    ax[1].axhline(0, color='k', ls='--')

ax[0].set_ylabel('$\Delta \log_{10}(\\mathrm{SMF})$')
ax[0].set_xlabel('Stellar Mass $M_*[M_\odot]$')
ax[1].set_xlabel('Stellar Mass $M_*[M_\odot]$')

ax[0].set_title('Training Data [1-40]')
ax[1].set_title('Validation Data [41-100]')

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150, sharey=True)
    
    plt.subplots_adjust(wspace=0, hspace=0)

    # Get upper and lower confidence bounds
    lower1, upper1 = observed_pred1.confidence_region()
    lower2, upper2 = observed_pred2.confidence_region()
    # Plot training data as black stars
    ax[0].axhline(1, color='k', ls='--')
    ax[1].axhline(1, color='k', ls='--')

    for i in range(40):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x1 = torch.asarray(pars(train_sel[i], np.log10(smf[train_sel[i]]['mstar'][:-5])), dtype=torch.float)
            observed_pred1 = likelihood(model(test_x1))
            lower1, upper1 = observed_pred1.confidence_region()
    
        ax[0].plot(np.log10(smf[train_sel[i]]['mstar'][:-5]),  smf[train_sel[i]]['smf'][:-5] / 10**observed_pred1.mean.numpy(), color='b', ls='', marker='o', ms=1)

    for i in range(60):
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            test_x2 = torch.asarray(pars(test_sel[i], np.log10(smf[test_sel[i]]['mstar'][:-4])), dtype=torch.float)
            observed_pred2 = likelihood(model(test_x2))
            lower2, upper2 = observed_pred2.confidence_region()
        
        ax[1].plot(np.log10(smf[test_sel[i]]['mstar'][:-4]), smf[test_sel[i]]['smf'][:-4] / 10**observed_pred2.mean.numpy(), color='r', ls='', marker='o', ms=1)

ax[0].set_ylabel('SMF Ratio')
ax[0].set_xlabel('Stellar Mass $M_*[M_\odot]$')
ax[1].set_xlabel('Stellar Mass $M_*[M_\odot]$')

ax[0].set_title('Training Data [1-40]')
ax[1].set_title('Validation Data [41-100]')